# Downloading Data from the Abacus Dataverse

I will be retrieving monthly Canadian Labour Force Survey data public files from the Abacus Dataverse hosted by the University of British Columbia library (https://abacus.library.ubc.ca/).
While the usual way to access data from the repository is graphically using its point-and-click web interface, all Dataverses are also able to serve their files through API (https://guides.dataverse.org/en/5.6/).

My intention is to eventually download the entire available history of LFS data to do some example analysis.
Hence it is neither economical nor easily replicable to use the point-and-click interface.
This notebook will be a demonstration of how to get metadata and request (publicly-available) files from the Dataverse server.

In [1]:
import requests
import json
import re
import time
import pandas as pd
import pymongo
import datetime

We set up some global variables here to make life easy later on.

In [2]:
endpoints = {
    'search': 'https://abacus.library.ubc.ca/api/search',
    'metadata': 'https://abacus.library.ubc.ca/api/datasets/:persistentId',
    'file_download': 'https://abacus.library.ubc.ca/api/access/datafile/:persistentId'
}

# 1. Search for Datasets

First, we need to retrieve metadata of relevant datasets to fetch later.

In [3]:
search_params = {
    'q': 'title:Labour Force Survey',
    'type': 'dataset',
    'per_page': 100
}
response = requests.get(url=endpoints['search'], params=search_params)
print('Response Status:', response.status_code)

Response Status: 200


We know from the documentation that there is per-page rate limiting, so we will recover the full list of relevant results through a simple loop that accounts for the possibility of failure due to accidentally going over the rate limit or otherwise.

In [4]:
items = response.json()['data']['items'].copy()
for start in range(search_params['per_page'], response.json()['data']['total_count']+1, search_params['per_page']):
    search_params['start'] = start
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['search'], params=search_params)
        if response.status_code == 200:
            items.extend(response.json()['data']['items'])
            fail_response = False
            print('Success:', start)
            time.sleep(.1)
        else:
            print('Fail:', start)
            time.sleep(5)
print('Total Returned Items:', len(items))

Success: 100
Success: 200
Success: 300
Success: 400
Success: 500
Success: 600
Success: 700
Success: 800
Total Returned Items: 892


We then use regex to recover the appropriate files.
Canada uses both American and British spelling and the Labour Force Survey is sometimes also abbreviated as the LFS, so regex is appropriate here.

In [5]:
lfs_re = re.compile(r'(?i:labou?r force survey|lfs)')
lfs_items = list(filter(lambda i: lfs_re.match(i['name']), items))
lfs_items.sort(key=lambda i: i['name'], reverse=True)
print('Labour Force Survey datasets:', len(lfs_items))

Labour Force Survey datasets: 62


Before writing the data, do some simple transformations that will save a lot of time and work later.
First is to convert datetime strings to datetime format.

In [6]:
for i in range(len(lfs_items)):
    for f in ['published_at', 'createdAt', 'updatedAt']:
        lfs_items[i][f] = datetime.datetime.strptime(lfs_items[i][f], r'%Y-%m-%dT%H:%M:%SZ')

Second is to recover the LFS series number (i.e. the Year number).
This one is important because previous surveys in the program have been updated and republished since their original publication.
We generally want the latest versions because StatCan has fixed data issues and incorrect survey weights.

In [7]:
yr_re = re.compile(r'\,\s*(\d{4})\W*$')
for i in range(len(lfs_items)):
    lfs_items[i]['seriesNum'] = int(yr_re.search(lfs_items[i]['name']).group(1))

I need a reference file so that I do not keep disturbing the API unnecessarily later.
After visually inspecting the list briefly to be sure that this is what I want, I will export the saved results to a MongoDB database.

In [8]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_col_datasets = lfs_db['datasets']

Pinged your deployment. You successfully connected to MongoDB!


Now we can upload the metadata using the Python MongoDB Driver API.

In [9]:
new_ids = lfs_col_datasets.insert_many(lfs_items).inserted_ids
print(len(new_ids))
print(new_ids[:5])

62
[ObjectId('6470cfd3661078467fabae4f'), ObjectId('6470cfd3661078467fabae50'), ObjectId('6470cfd3661078467fabae51'), ObjectId('6470cfd3661078467fabae52'), ObjectId('6470cfd3661078467fabae53')]


The Driver makes it quite convenient now to retrieve the specific fields we will need going forward.
I also make use of its aggregator pipeline functionality to recover only the versions of interest of republished LFS's.
Below, briefly, I use the `$group` aggregator function to get the unique dataset for each year (`seriesNum`) that has the highest `versionId`.

In [10]:
pids = list(lfs_col_datasets.aggregate([
    {'$group': {
        '_id': '$seriesNum',
        'name': {
            '$top': {
                'output': '$name',
                'sortBy': {'versionId': -1}
            }
        },
        'global_id': {
            '$top': {
                'output': '$global_id',
                'sortBy': {'versionId': -1}
            }
        }
    }},
    {'$sort': {
        '_id': -1
    }}
]))

Things are done.
We can close the connection to the database.

In [11]:
client.close()

# 2. Get Dataset Metadata

We now call a second API to recover the metadata and, especially, the file list of each dataset.

In [12]:
search_params = {
    'persistentId': pids[0]['global_id']
}
response = requests.get(url=endpoints['metadata'], params=search_params)
print('Response Status:', response.status_code)

Response Status: 200


The metadata of interest are stored in the `files` subsection.

In [13]:
items = response.json()['data']['latestVersion']['files']

We are going to loop through the remaining data sets and compile the datafiles list.

In [14]:
for ds in pids[1:]:
    search_params['persistentId'] = ds['global_id']
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['metadata'], params=search_params)
        if response.status_code == 200:
            items.extend(response.json()['data']['latestVersion']['files'])
            fail_response = False
            print('Success:', ds)
            time.sleep(.1)
        else:
            print('Fail:', ds)
            time.sleep(5)
print('Total Returned Items:', len(items))

Success: {'_id': 2022, 'name': 'Labour Force Survey, 2022', 'global_id': 'hdl:11272.1/AB2/SRAU7E'}
Success: {'_id': 2021, 'name': 'Labour Force Survey, 2021', 'global_id': 'hdl:11272.1/AB2/HP9TEK'}
Success: {'_id': 2020, 'name': 'Labour Force Survey, 2020', 'global_id': 'hdl:11272.1/AB2/GGXMM2'}
Success: {'_id': 2019, 'name': 'Labour Force Survey, 2019', 'global_id': 'hdl:11272.1/AB2/ATGWRX'}
Success: {'_id': 2018, 'name': 'Labour Force Survey, 2018', 'global_id': 'hdl:11272.1/AB2/CFRSC0'}
Success: {'_id': 2017, 'name': 'Labour Force Survey, 2017', 'global_id': 'hdl:11272.1/AB2/PO8CWI'}
Success: {'_id': 2016, 'name': 'Labour Force Survey, 2016', 'global_id': 'hdl:11272.1/AB2/SRHMXU'}
Success: {'_id': 2015, 'name': 'Labour Force Survey, 2015', 'global_id': 'hdl:11272.1/AB2/N1N5S6'}
Success: {'_id': 2014, 'name': 'Labour Force Survey, 2014', 'global_id': 'hdl:11272.1/AB2/B9RMKT'}
Success: {'_id': 2013, 'name': 'Labour Force Survey, 2013', 'global_id': 'hdl:11272.1/AB2/1CUHBA'}
Success: {

As before, we will save the data files metadata into the MongoDB database for quick retrieval in the future.

In [3]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_col_datafiles = lfs_db['datafiles']

Pinged your deployment. You successfully connected to MongoDB!


In [16]:
new_ids = lfs_col_datafiles.insert_many(items).inserted_ids
print(len(new_ids))
print(new_ids[:5])

782
[ObjectId('6470d028661078467fabae8e'), ObjectId('6470d028661078467fabae8f'), ObjectId('6470d028661078467fabae90'), ObjectId('6470d028661078467fabae91'), ObjectId('6470d028661078467fabae92')]


In [4]:
fids = list(lfs_col_datafiles.find(filter={
    'dataFile.filename': {
        '$regex': r'\.(?i:tab)$'
    },
    'dataFile.description': {
        '$regex': r'(?i:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|ann)\w*\s+(?i:\d+\s+data|data\s+\d+)'
    }
}, projection={
    'dataFile': {
        'persistentId': 1,
        'filename': 1,
        'contentType': 1,
        'description': 1
    }
}))
print('Items Retrieved:', len(fids))

Items Retrieved: 268


Close the client once done.

In [11]:
client.close()

# 3. Download Identified Files

We can now request for only the specific files identified from the metadata search.
Because harmonisation is actually needed before all files can be combined, it is necessary to save files locally first for further reprocessing before uploading to a database.

In [13]:
filename_re = re.compile(r'^(?P<pref>\w+)_(?P<mth>0?\d|1[012]|(?i:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\w*)_(?P<yr>\d{4})\.(?P<ext>\w+)$', flags=re.I)
month_map = {
    'jan': '01',
    'feb': '02',
    'mar': '03',
    'apr': '04',
    'may': '05',
    'jun': '06',
    'jul': '07',
    'aug': '08',
    'sep': '09',
    'oct': '10',
    'nov': '11',
    'dec': '12',
    'january': '01',
    'february': '02',
    'march': '03',
    'april': '04',
    'may': '05',
    'june': '06',
    'july': '07',
    'august': '08',
    'september': '09',
    'october': '10',
    'november': '11',
    'december': '12'
}
request_result = dict()
for i in fids:
    data_file = i['dataFile']
    data_file['num_fails'] = 0
    fn_part = filename_re.match(data_file['filename'].lower()).groupdict()
    fn_part['mth'] = month_map[fn_part['mth']] if fn_part['mth'] in month_map else fn_part['mth']
    fn = '{pref}-{yr}-{mth}.{ext}'.format(**fn_part)
    data_file['surv_yr'] = int(fn_part['yr'])
    data_file['surv_mth'] = int(fn_part['mth'])
    search_params = {
        'persistentId': data_file['persistentId']
    }
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['file_download'], params=search_params)
        if response.status_code == 200:
            file_loc = 'data/{}'.format(fn)
            with open(file_loc, 'tw') as f:
                f.write(response.text)
            data_file['num_obs'] = pd.read_csv(file_loc, sep='\t', usecols=[0]).index.size
            fail_response = False
            time.sleep(.1)
        else:
            data_file['num_fails'] = data_file['num_fails'] + 1
            time.sleep(5)
    request_result[fn] = data_file
request_result = pd.DataFrame.from_dict(request_result, orient='index')

Show some information about the download process

In [14]:
request_result.head(5)

,persistentId,filename,contentType,description,num_fails,surv_yr,surv_mth,num_obs
lfs-2023-04.tab,hdl:11272.1/AB2/IJU1QK/GTMJCJ,LFS_April_2023.tab,text/tab-separated-values,April 2023 data (subsettable) (original SPSS ....,0,2023,4,100024
lfs-2023-02.tab,hdl:11272.1/AB2/IJU1QK/SC0TCM,LFS_February_2023.tab,text/tab-separated-values,February 2023 data (subsettable) (original SPS...,0,2023,2,104851
lfs-2023-01.tab,hdl:11272.1/AB2/IJU1QK/4RZ0VX,LFS_January_2023.tab,text/tab-separated-values,January 2023 data (subsettable) (original SPSS...,0,2023,1,108064
lfs-2023-03.tab,hdl:11272.1/AB2/IJU1QK/ZZVBUB,LFS_March_2023.tab,text/tab-separated-values,March 2023 data (subsettable) (original SPSS ....,0,2023,3,101043
lfs-2022-04.tab,hdl:11272.1/AB2/SRAU7E/ERMNLM,LFS_April_2022.tab,text/tab-separated-values,April 2022 data (subsettable) (original SPSS ....,0,2022,4,111708


In [15]:
request_result.tail(5)

,persistentId,filename,contentType,description,num_fails,surv_yr,surv_mth,num_obs
lfs-2001-08.tab,hdl:11272.1/AB2/MKMPL2/S57UWT,lfs_08_2001.tab,text/tab-separated-values,August 2001 data (subsettable),0,2001,8,104499
lfs-2001-09.tab,hdl:11272.1/AB2/MKMPL2/2ZLULH,lfs_09_2001.tab,text/tab-separated-values,September 2001 data (subsettable),0,2001,9,105012
lfs-2001-10.tab,hdl:11272.1/AB2/MKMPL2/5PW3AQ,lfs_10_2001.tab,text/tab-separated-values,October 2001 data (subsettable),0,2001,10,105608
lfs-2001-11.tab,hdl:11272.1/AB2/MKMPL2/ZWJOXF,lfs_11_2001.tab,text/tab-separated-values,November 2001 data (subsettable),0,2001,11,105516
lfs-2001-12.tab,hdl:11272.1/AB2/MKMPL2/HUO9IL,lfs_12_2001.tab,text/tab-separated-values,December 2001 data (subsettable),0,2001,12,105115


In [16]:
request_result.to_csv('data/request_result_lfs.csv', index=True, encoding='utf-8-sig')

# 4. Find the Codebooks

The tab-delimited files are almost purely numeric.
However, to properly interpret the variables it is also necessary to retrieve the codebooks to understand how to code these variables.

In [ ]:
_mongo_details = json.load(open('.mongo.json'))
_mongo_url = 'mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority'.format(*_mongo_details.values())
client = pymongo.mongo_client.MongoClient(_mongo_url, server_api=pymongo.server_api.ServerApi('1'))
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)
lfs_db = client['lfs']
lfs_col_datafiles = lfs_db['datafiles']

Briefly, what goes on below is:
1. Find a unique match by file name (`dataFile.filename`).
The unique match that we want is the last uploaded version of the file, which can be detected using the data set id (`dataserVersionId`).
2. Return only SPSS syntax files.
3. Sort results in descending order by the file name.

We want specifically only the SPSS syntax files because from some other investigations not documented here, only the SPSS syntax files for loading fixed-width ASCII files and the user guides have the value labels (the syntax files technically do not comprise the codebook because it of course does not describe things like the frequency of each value).

Additionally, SPSS syntax files are conveniently just text files with a different extension with information laid out in a fairly consistent format, like most programming script files are.
This makes it very amenable to text processing and data extraction using another programming language.

In [ ]:
fids = list(lfs_col_datafiles.aggregate([
    {'$group': {
        '_id': '$dataFile.filename',
        'persistentId': {
            '$top': {
                'output': '$dataFile.persistentId',
                'sortBy': {'datasetVersionId': -1}
            }
        },
        'description': {
            '$top': {
                'output': '$dataFile.description',
                'sortBy': {'versionId': -1}
            }
        }
    }},
    {'$match': {
        '_id': {
            '$regex': r'\.(?i:spss?)$'
        }
    }},
    {'$sort': {
        '_id': -1
    }}
]))

Close the client once done.

In [ ]:
client.close()

Download the files with scope for retry.

In [ ]:
request_result = dict()
for i in fids:
    fn = i['_id'][:]
    fn = re.sub(r'^[a-z]+[-_]+', '', fn, flags=re.I)
    fn = re.sub(r'\.[a-z]+$', '', fn, flags=re.I)
    fn_part = dict()
    if re.match(r'^rebase', fn, flags=re.I):
        rebase_re = re.compile(r'^rebase[-_a-z]+(\d{4})', flags=re.I)
        fn_part['rebase'] = rebase_re.match(fn).group(1)
        fn = rebase_re.sub('', fn)
    elif re.match(r'.+?rebase', fn, flags=re.I):
        rebase_re = re.compile(r'(.+?)rebase[-_a-z]+(\d{4})', flags=re.I)
        fn_part['rebase'] = rebase_re.match(fn).group(2)
        fn = rebase_re.sub(r'\1', fn)
    else:
        continue
    fn = re.sub(r'(^[^a-z0-9]+|[^a-z0-9]$)', '', fn, flags=re.I)
    period_re = re.compile(r'((?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|\d{4})[-_a-z0-9]*(?:[-_]+\d{4})?)', flags=re.I)
    fn_part['period'] = period_re.match(fn).group(1)
    fn = period_re.sub('', fn)
    period_re = re.compile(r'((?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*)[-_]+([a-z0-9][-_a-z0-9]+[a-z0-9])', flags=re.I)
    while period_re.match(fn_part['period']):
        fn_part['period'] = period_re.sub(r'\2-\1', fn_part['period'])
    fn = 'syntax_{period}_rebase{rebase}.txt'.format(**fn_part)
    #   The rest of it requests the syntax files
    i['num_fails'] = 0
    search_params = {
        'persistentId': i['persistentId']
    }
    fail_response = True
    while fail_response:
        response = requests.get(url=endpoints['file_download'], params=search_params)
        if response.status_code == 200:
            file_loc = 'data/{}'.format(fn)
            with open(file_loc, 'tw') as f:
                f.write(response.text)
            fail_response = False
            time.sleep(.1)
        else:
            i['num_fails'] = i['num_fails'] + 1
            time.sleep(5)
    del i['_id']
    request_result[fn] = i
request_result = pd.DataFrame.from_dict(request_result, orient='index')

Show some information about the download process.

In [ ]:
request_result

,persistentId,description,num_fails
syntax_1999-nov-dec_rebase2001.txt,hdl:11272.1/AB2/LGGEKU/FI1VWQ,SPSS syntax November-December 1999,0
syntax_1999-jan-oct_rebase2001.txt,hdl:11272.1/AB2/LGGEKU/FFZGU0,SPSS syntax January-October 1999,0
syntax_2000-2010_rebase2001.txt,hdl:11272.1/AB2/F0G05L/DKBI2F,SPSS syntax 2000-2010,0
syntax_1987-1998_rebase2001.txt,hdl:11272.1/AB2/PW64LJ/MVVTLK,SPSS syntax 1987-1998,0
syntax_1976-1986_rebase2001.txt,hdl:11272.1/AB2/SS4B9S/RU28Z5,SPSS syntax 1971-1986,0
syntax_2022_rebase2023.txt,hdl:11272.1/AB2/SRAU7E/S0JZL7,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2021_rebase2023.txt,hdl:11272.1/AB2/HP9TEK/7UJJRS,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2020_rebase2023.txt,hdl:11272.1/AB2/GGXMM2/X7TBKH,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2019_rebase2023.txt,hdl:11272.1/AB2/ATGWRX/K75CHT,SPSS syntax for plain-text microdata files. Cr...,0
syntax_2018_rebase2023.txt,hdl:11272.1/AB2/CFRSC0/GSGTYS,SPSS syntax for plain-text microdata files. Cr...,0


In [ ]:
request_result.to_csv('data/request_result_syntax.csv', index=True, encoding='utf-8-sig')

# Summary

In this first step, we downloaded and stored some data of the Canadian Labour Force Surveys from the Abacus data repository hosted by the University of British Columbia.
The information includes:
1. Metadata of LFS-related data sets, which has been uploaded to a MongoDB database.
2. Metadata of LFS-related data files, which has been uploaded to a MongoDB database.
3. Tab-delimited data files for 2023 to 2001, which are currently stored locally for processing before deciding whether to upload to a relational database.
4. SPSS syntax files, which are currently stored locally for processing.